In [49]:
# Convert a torch-distributed-checkpoint to plain torch checkpoint
# This will generate a '<student/teacher>-torch.pth'
# IMPORTANT: Do not specify a *.distcp file, instead specify the root directory.
#
# Example: sh smiths/distcp2pth.py outputs/hif_dinov3/ckpt/999 /path/to/output/dir
#
# Author: Soumen Sardar
# Date: 2026-02-24

import os
import pathlib
import argparse
from dinov3.train.ssl_meta_arch import SSLMetaArch
import torch
from omegaconf import OmegaConf
from contextlib import suppress

input_path = "outputs/privatedata_dinov3-small/ckpt/399"
output_dir = "/home/icmore_acc/Downloads/pyvision_dev/pytorch_vision"

# load the config from the checkpoint directory
cfg_file = pathlib.Path(input_path).parents[1] / "config.yaml"
cfg = OmegaConf.load(cfg_file)

model_arch = cfg.student.arch
dataset = cfg.train.dataset_path.split(":")[0]

# create output files
pth_interm_file = pathlib.Path(output_dir) / f"dinov3_{model_arch}_{dataset}_intermidiate.pth"
pth_student_file = pathlib.Path(output_dir) / f"dinov3_{model_arch}_{dataset}_student.pth"
pth_teacher_file = pathlib.Path(output_dir) / f"dinov3_{model_arch}_{dataset}_teacher.pth"
total_steps = 6

In [52]:
print(f"[1 / {total_steps}] Removing the intermediate torch checkpoint...")
with suppress(FileNotFoundError): os.remove(pth_interm_file)
print(f"[2 / {total_steps}] Converting distributed to intermidiate torch checkpoint form '{input_path}'")
os.system(r"python -m torch.distributed.checkpoint.format_utils dcp_to_torch %s %s" % (input_path, str(pth_interm_file)))
print("DONE")

[1 / 6] Removing the intermediate torch checkpoint...
[2 / 6] Converting distributed to intermidiate torch checkpoint form 'outputs/privatedata_dinov3-small/ckpt/399'
Converting checkpoint from outputs/privatedata_dinov3-small/ckpt/399 to /home/icmore_acc/Downloads/pyvision_dev/pytorch_vision/dinov3_vit_small_PRIVATE_DATA_intermidiate.pth using method: 'dcp_to_torch'
DONE


In [53]:
print(f"[3 / {total_steps}] Loading intermidiate torch checkpoint...")
state_dict = torch.load(pth_interm_file, weights_only=False)

# with torch.device("meta"):
model = SSLMetaArch(cfg)
model.to_empty(device='cpu')
model.load_state_dict(state_dict['model'], strict=True) # load checkpoint
# model.to_empty(device='cpu') # To fix error: 'Cannot copy out of meta tensor; no data!' 

[3 / 6] Loading intermidiate torch checkpoint...


<All keys matched successfully>

In [54]:
for k, v in state_dict['model'].items():
    if "student.backbone." in k:
        print(k, v.is_meta, v.flatten()[0])

student.backbone.cls_token False tensor(-0.0053)
student.backbone.storage_tokens False tensor(0.0060)
student.backbone.mask_token False tensor(-0.0011)
student.backbone.patch_embed.proj.weight False tensor(-0.0203)
student.backbone.patch_embed.proj.bias False tensor(0.0322)
student.backbone.blocks.0.norm1.weight False tensor(1.0003)
student.backbone.blocks.0.norm1.bias False tensor(0.0002)
student.backbone.blocks.0.attn.qkv.weight False tensor(0.0088)
student.backbone.blocks.0.attn.qkv.bias False tensor(-0.0002)
student.backbone.blocks.0.attn.proj.weight False tensor(0.0061)
student.backbone.blocks.0.attn.proj.bias False tensor(-0.0006)
student.backbone.blocks.0.ls1.gamma False tensor(0.0024)
student.backbone.blocks.0.norm2.weight False tensor(1.0012)
student.backbone.blocks.0.norm2.bias False tensor(-0.0010)
student.backbone.blocks.0.mlp.fc1.weight False tensor(-0.0317)
student.backbone.blocks.0.mlp.fc1.bias False tensor(-0.0016)
student.backbone.blocks.0.mlp.fc2.weight False tensor(0

In [64]:
if True:
    print(f"[4 / {total_steps}] Saving the student and teacher BACKBONE weights to separate torch checkpoint files...")
    with suppress(FileNotFoundError): os.remove(pth_student_file)
    with suppress(FileNotFoundError): os.remove(pth_teacher_file)
    torch.save(model.student.backbone.state_dict(), pth_student_file)
    torch.save(model.teacher.backbone.state_dict(), pth_teacher_file)

else:
    def search_func(backbone_name, key):
        return backbone_name in key
            # if "local_cls_norm" in key:
            #     return False
            # else:
                # return True
        return False
    
    search_key = "student.backbone."
    student_dict = {k.replace(search_key, ""):v for k, v in state_dict['model'].items() if search_func(search_key, k)}
    
    search_key = "teacher.backbone."
    teacher_dict = {k.replace(search_key, ""):v for k, v in state_dict['model'].items() if search_func(search_key, k)}
    
    print(f"[4 / {total_steps}] Saving the student and teacher DICT weights to separate torch checkpoint files...")
    with suppress(FileNotFoundError): os.remove(pth_student_file)
    with suppress(FileNotFoundError): os.remove(pth_teacher_file)
    torch.save(student_dict, pth_student_file)
    torch.save(teacher_dict, pth_teacher_file)
print("DONE")

[4 / 6] Saving the student and teacher BACKBONE weights to separate torch checkpoint files...
DONE


In [65]:

# student_dict = torch.load(pth_student_file, weights_only=False)
# teacher_dict = torch.load(pth_teacher_file, weights_only=False)
# with suppress(FileNotFoundError): os.remove(pth_student_file)
# with suppress(FileNotFoundError): os.remove(pth_teacher_file)
# torch.save(student_dict, pth_student_file)
# torch.save(teacher_dict, pth_teacher_file)

In [66]:
# state_dict = torch.load(pth_student_file, weights_only=False)
# # for k, v in state_dict.items():
# #     if v.is_meta:
# #         print(k, v.is_meta, v.flatten()[0])
# #     if "local_cls_norm" in k:
# #         print(k)
# with suppress(FileNotFoundError): os.remove("student2.pth")
# torch.save(student_dict, "student2.pth")

'/home/icmore_acc/.cache/torch/hub'

In [70]:
print(f"[5 / {total_steps}] verifying checkpoint...", end="")

# verify the generated checkpoints
cache_dir = torch.hub.get_dir()
with suppress(FileNotFoundError): os.remove(pathlib.Path(cache_dir) / os.path.basename(pth_student_file))
verify_model = torch.hub.load(
    ".",
    "dinov3_vits16",
    weights=str(pth_student_file),
    source="local",
)
# verify the generated checkpoints
verify_model = torch.hub.load(
    ".",
    "dinov3_vits16",
    weights=str(pth_teacher_file),
    source="local",
    force_reload=True,
)
print("DONE")

[5 / 6] verifying checkpoint...

RuntimeError: Error(s) in loading state_dict for DinoVisionTransformer:
	Unexpected key(s) in state_dict: "local_cls_norm.weight", "local_cls_norm.bias". 

In [ ]:
"""
# 18 -> 8.65 (8.5) [1999+18% = 2360], [stampduty=kolkata 100+18%], (fore-close 24m | part-payment 3%)
# 15 -> aadhaar, pan, current proof
(8.35 upto 7yrs)
"""